# SpikeInterface workflow for NP recordings
### https://spikeinterface.readthedocs.io/en/latest/index.html 

#### NP Processing Pipeline: 
1. run local catGT
2. move file to server and to this local computer
3. run this notebook for ks2.5 and QC (you only have to specify folder and tags)
4. manual curation with Phy

In [1]:
import os
# if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")

import pandas as pd
import spikeinterface.full as si

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from spikeinterface import create_sorting_analyzer, load_sorting_analyzer


In [10]:
session_name = '20251210-SL412-Reward4_g0'

# Read data
exp_name = '202512-Ephys-LHb'
session_path = rf'/Users/shunli/Projects/pyNeuroDAP/Data/{session_name}'
spike_path = os.path.join(session_path, rf'AIND_{session_name}')
nwb_path = os.path.join(session_path, 'nwb','ecephys_session_block0_recording1.nwb')
save_folder = os.path.join(session_path, 'results')

spikeglx_folder_imec0 = Path(os.path.join(session_path, rf'{session_name}_imec0'))
spikeglx_folder_imec1 = Path(os.path.join(session_path, rf'{session_name}_imec1'))

print('Analyzing session: ', session_name)
print('Session path: ', session_path)
print('Spike path: ', spike_path)
print('Save folder: ', save_folder)

Analyzing session:  20251210-SL412-Reward4_g0
Session path:  /Users/shunli/Projects/pyNeuroDAP/Data/20251210-SL412-Reward4_g0
Spike path:  /Users/shunli/Projects/pyNeuroDAP/Data/20251210-SL412-Reward4_g0/AIND_20251210-SL412-Reward4_g0
Save folder:  /Users/shunli/Projects/pyNeuroDAP/Data/20251210-SL412-Reward4_g0/results


In [11]:
# Check if both imec0 and imec1 folders exist
if spikeglx_folder_imec0.exists() and spikeglx_folder_imec1.exists():
    # output_folder_0 = Path(rf"{session_path}\output_0")
    # output_folder_1 = Path(rf"{session_path}\output_1")
    # output_folder_0.mkdir(exist_ok=True, parents=True)
    # output_folder_1.mkdir(exist_ok=True, parents=True)
    print("Both imec0 and imec1 folders found. Output folders created.")
elif spikeglx_folder_imec0.exists():
    output_folder = session_path
    spikeglx_folder = os.path.join(session_path, rf'{session_name}_imec0')
elif spikeglx_folder_imec1.exists():
    output_folder = session_path
    spikeglx_folder = os.path.join(session_path, rf'{session_name}_imec1')
    
dup_removed = 0 # or 1
skipOptional = 1 # skip plotting and drift checking before running kilosort

In [16]:
spikeglx_folder_imec0

PosixPath('/Users/shunli/Projects/pyNeuroDAP/Data/20251210-SL412-Reward4_g0/20251210-SL412-Reward4_g0_imec0')

In [13]:
folders = []
if spikeglx_folder_imec0.exists():
    folders.append((spikeglx_folder_imec0, 'imec0.ap'))
if spikeglx_folder_imec1.exists():
    folders.append((spikeglx_folder_imec1, 'imec1.ap'))

for folder, stream_name in folders:
    raw_rec = si.read_spikeglx(folder, stream_name=stream_name, load_sync_channel=False)
    noise_levels_microV = si.get_noise_levels(raw_rec, return_scaled=True)
    
    fig, ax = plt.subplots()
    _ = ax.hist(noise_levels_microV)
    ax.set_xlabel('noise [microV]')
    ax.set_ylabel('count')
    title = f'Noise levels for {session_name}_{stream_name}'
    ax.set_title(title)
    # save_path = output_folder / f'noise_levels_{stream_name}.tif'
    # plt.savefig(save_path)
    plt.show()

## Read in the data and find data streams within the data folder

In [ ]:
stream_names, stream_ids = si.get_neo_streams('spikeglx', spikeglx_folder)
stream_names

In [ ]:
# we do not load the sync channel, so the probe is automatically loaded
raw_rec = si.read_spikeglx(spikeglx_folder, stream_name='imec0.ap', load_sync_channel=False)
raw_rec

In [ ]:
# rec = raw_rec
# a = rec.get_traces()
# a.shape
# np.max(a[1:10,:])

In [ ]:
# rec = raw_rec
# # we can estimate the noise on the scaled traces (microV) or on the raw one (which is in our case int16).
# noise_levels_microV = si.get_noise_levels(rec, return_scaled=True, method = 'std')
# # noise_levels_microV = si.get_noise_levels(rec, return_scaled=True)
# noise_levels_int16 = si.get_noise_levels(rec, return_scaled=False, method = 'std')

In [ ]:
# fig, ax = plt.subplots()
# _ = ax.hist(noise_levels_int16)
# ax.set_xlabel('noise  [int16]')

In [ ]:
# fig, ax = plt.subplots()
# _ = ax.hist(noise_levels_microV)
# ax.set_xlabel('noise  [microV]')

In [ ]:
# np.median(noise_levels_int16)

## View your probe geometry

In [ ]:
# we automaticaly have the probe loaded!
pd.set_option('display.max_rows', 10)
df = raw_rec.get_probe().to_dataframe()

fig, ax = plt.subplots(figsize=(60, 40))
si.plot_probe_map(raw_rec, ax=ax, with_channel_ids=True)
# ax.set_ylim(-200, 750)

## Preprocess the recording and identify bad channels

In [ ]:
#rec1 = si.highpass_filter(raw_rec, freq_min=400.)
#bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)
#rec2 = rec1.remove_channels(bad_channel_ids)
#print('bad_channel_ids', bad_channel_ids)

#rec3 = si.phase_shift(rec2) 
#rec4 = si.common_reference(rec3, operator="median", reference="global")
#rec = rec4
#rec

## 20251112 Bad Channel Debug

In [ ]:
import inspect
import spikeinterface.preprocessing as spre
print("SI version:", si.__version__)
print(inspect.signature(spre.detect_bad_channels))
help(spre.detect_bad_channels)  # prints the docstring with defaults


In [ ]:
# rec = rec1

# # See what SI thinks
# print("is_filtered annotation:", getattr(rec, "is_filtered", None))          # some versions
# try:
#     print("annotation:", rec.get_annotation("is_filtered"))
# except Exception:
#     pass

# # If you *know* it is already high-pass/band-pass filtered, mark it:
# # rec = rec.annotate(is_filtered=True)


In [ ]:
rec1 = si.highpass_filter(raw_rec, freq_min=400.)
#bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)
#rec2 = rec1.remove_channels(bad_channel_ids)
#print('bad_channel_ids', bad_channel_ids)


rec3 = si.phase_shift(rec1) 
rec4 = si.common_reference(rec3, operator="median", reference="global")

In [ ]:
# #bandpass filter
# import spikeinterface.preprocessing as spre

# # --- Bandpass filter to remove low and high frequency noise ---
# rec1 = spre.bandpass_filter(raw_rec, freq_min=400, freq_max=6000)

# # (Optional) phase alignment and referencing
# rec2 = spre.phase_shift(rec1)
# rec3 = spre.common_reference(rec2, operator="median", reference="global")

In [ ]:
rec = rec1

rec.has_scaleable_traces()
rec.get_channel_gains()[:10]
print(rec.has_scaleable_traces(), rec.get_channel_gains()[:8], rec.get_channel_offsets()[:8])



In [ ]:
print("is_filtered annotation:", getattr(rec, "is_filtered", None))          # some versions
try:
    print("annotation:", rec.get_annotation("is_filtered"))
except Exception:
    pass

In [ ]:
# method_use = 'neighborhood_r2'
method_use = 'coherence+psd'

bad_channel_ids, channel_labels = si.detect_bad_channels(rec, method=method_use)
rec2 = rec1.remove_channels(bad_channel_ids)

print('method', method_use)
print('bad_channel_ids', bad_channel_ids)
print('channel_labels', channel_labels)

## Diagnostic plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import spikeinterface as si
import spikeinterface.preprocessing as spre

# Internals used by SI
from spikeinterface.core import get_random_data_chunks, order_channels_by_depth


# -----------------------------
# Config (match your SI call)
# -----------------------------
method = "coherence+psd"
psd_hf_threshold      = 0.02   # uV^2/Hz
dead_channel_thr      = -0.5
noisy_channel_thr     = 1.0
outside_channel_thr   = -0.75
n_neighbors           = 11
nyquist_threshold     = 0.8    # >80% Nyquist
welch_window_ms       = 10.0   # ms (outer function default)
chunk_duration_s      = 0.3
num_random_chunks     = 100
seed                  = 0
outside_channels_location = "top"   # "top" | "bottom" | "both"

# set rec above
# rec = rec4  # <- same object you pass to spre.detect_bad_channels

# -----------------------------
# Helpers copied from SI
# -----------------------------
def detrend(x, nmed):
    """Median-filter trend removal (IBL helper)."""
    import scipy.signal
    ntap = int(np.ceil(nmed / 2))
    xf = np.r_[np.zeros(ntap) + x[0], x, np.zeros(ntap) + x[-1]]
    xf = scipy.signal.medfilt(xf, nmed)[ntap:-ntap]
    return x - xf

def ibl_chunk_metrics(raw, fs, *,
                      psd_hf_threshold, dead_channel_thr, noisy_channel_thr, outside_channel_thr,
                      n_neighbors, nyquist_threshold, welch_window_ms, outside_channels_location):
    """Compute IBL intermediates for one chunk (raw: samples x channels)."""
    import scipy.signal
    raw = raw - np.mean(raw, axis=0, keepdims=True)
    nperseg = int(welch_window_ms * fs / 1000.0)
    fscale, psd = scipy.signal.welch(raw, fs=fs, axis=0, window="hann", nperseg=nperseg)

    # “coherence” proxies from IBL code
    ref = np.median(raw, axis=1)                                   # (samples,)
    xcorr = np.sum(raw * ref[:, None], axis=0) / np.sum(ref**2)    # (channels,)
    xcorr_neighbors = detrend(xcorr, n_neighbors)                   # local similarity
    xcorr_distant   = xcorr - detrend(xcorr, n_neighbors) - 1       # far similarity / tail

    # HF PSD mean above nyquist_threshold * Nyquist
    hf_start = (fs / 2.0) * nyquist_threshold
    hf_mask  = fscale > hf_start
    psd_hf   = np.mean(psd[hf_mask, :], axis=0)                     # uV^2 / Hz (if recording is scaled)

    # IBL label logic (0 good, 1 dead, 2 noise, 3 out)
    labels = np.zeros(raw.shape[1], dtype=int)
    idead  = np.where(xcorr_neighbors < dead_channel_thr)[0]
    inoisy = np.where(np.logical_or(psd_hf > psd_hf_threshold,
                                    xcorr_neighbors > noisy_channel_thr))[0]
    labels[idead] = 1
    labels[inoisy] = 2

    # “outside” based on distant coherence at ends
    (ioutside,) = np.where(xcorr_distant < outside_channel_thr)
    a = np.cumsum(np.r_[0, np.diff(ioutside) - 1])
    if ioutside.size > 0:
        if outside_channels_location == "top":
            if ioutside[-1] == (raw.shape[1] - 1):
                ioutside2 = ioutside[(a == np.max(a)) & (a > 0)]
                labels[ioutside2] = 3
        elif outside_channels_location == "bottom":
            if ioutside[0] == 0:
                ioutside2 = ioutside[(a == np.min(a)) & (a < np.max(a))]
                labels[ioutside2] = 3
        else:
            if ioutside[-1] == (raw.shape[1] - 1) or ioutside[0] == 0:
                ioutside2 = ioutside[(a == np.max(a)) | (a == np.min(a))]
                labels[ioutside2] = 3

    out = dict(
        fscale=fscale, psd=psd, psd_hf=psd_hf,
        xcorr=xcorr, xcorr_neighbors=xcorr_neighbors, xcorr_distant=xcorr_distant,
        labels=labels
    )
    return out

# -----------------------------
# Prepare recording like SI
# -----------------------------
# If not already filtered, SI applies a 300 Hz HP on the fly
rec_hp = rec if rec.is_filtered() else spre.highpass_filter(rec, freq_min=300)

# IMPORTANT: SI expects μV scaling for coherence+psd thresholds
try:
    assert rec_hp.has_scaleable_traces()
except Exception:
    # In >=0.99 the attribute is has_scaleable_traces(); in older it might differ
    pass

fs = rec_hp.get_sampling_frequency()

# Random chunks exactly like SI
random_chunk_kwargs = dict(
    num_chunks_per_segment=num_random_chunks,
    chunk_size=int(chunk_duration_s * fs),
    return_scaled=True,       # IMPORTANT for μV
    concatenated=False,
    seed=seed,
)
random_data = get_random_data_chunks(rec_hp, **random_chunk_kwargs)  # list of (samples x channels)

# Order channels by depth (so neighbor/distant logic matches SI)
order_f, order_r = order_channels_by_depth(recording=rec_hp, dimensions=("x", "y"))
if np.all(np.diff(order_f) == 1):  # already ordered
    order_f = order_r = None

# -----------------------------
# Compute intermediates per chunk
# -----------------------------
per_chunk = []
for chunk in random_data:
    chunk_sorted = chunk[:, order_f] if order_f is not None else chunk
    M = ibl_chunk_metrics(
        chunk_sorted, fs,
        psd_hf_threshold=psd_hf_threshold,
        dead_channel_thr=dead_channel_thr,
        noisy_channel_thr=noisy_channel_thr,
        outside_channel_thr=outside_channel_thr,
        n_neighbors=n_neighbors,
        nyquist_threshold=nyquist_threshold,
        welch_window_ms=welch_window_ms,
        outside_channels_location=outside_channels_location,
    )
    # unsort back to original channel order
    if order_r is not None:
        for k in ("psd_hf", "xcorr", "xcorr_neighbors", "xcorr_distant", "labels"):
            M[k] = M[k][order_r]
    per_chunk.append(M)

# -----------------------------
# Aggregate across chunks
# -----------------------------
# SI takes the MODE of labels across chunks; for continuous features we’ll take MEDIAN
labels_stack = np.stack([m["labels"] for m in per_chunk], axis=1)          # (nch, nchunks)
# mode without scipy (ties → first)
def simple_mode(a, axis=1):
    # a: int array (nch, nchunks)
    out = np.empty(a.shape[0], dtype=a.dtype)
    for i in range(a.shape[0]):
        vals, counts = np.unique(a[i], return_counts=True)
        out[i] = vals[np.argmax(counts)]
    return out
labels_mode = simple_mode(labels_stack)

psd_hf_med         = np.median(np.stack([m["psd_hf"] for m in per_chunk], axis=1), axis=1)
xcorr_neighbors_med= np.median(np.stack([m["xcorr_neighbors"] for m in per_chunk], axis=1), axis=1)
xcorr_distant_med  = np.median(np.stack([m["xcorr_distant"] for m in per_chunk], axis=1), axis=1)

# -----------------------------
# Print quick diagnostics
# -----------------------------
print(f"HF band starts at {nyquist_threshold} * Nyquist = {nyquist_threshold*(fs/2):.1f} Hz")
print(f"psd_hf_threshold: {psd_hf_threshold} μV^2/Hz")
print("psd_hf   (min/med/max):", np.min(psd_hf_med), np.median(psd_hf_med), np.max(psd_hf_med))
print("xcorr_neighbors (min/med/max):", np.min(xcorr_neighbors_med), np.median(xcorr_neighbors_med), np.max(xcorr_neighbors_med))
print("xcorr_distant   (min/med/max):", np.min(xcorr_distant_med), np.median(xcorr_distant_med), np.max(xcorr_distant_med))

unique, counts = np.unique(labels_mode, return_counts=True)
print("Mode labels counts {0:good,1:dead,2:noise,3:out}:", dict(zip(unique, counts)))

# Which rule(s) are actually firing?
noise_hf  = psd_hf_med > psd_hf_threshold
noise_coh = xcorr_neighbors_med > noisy_channel_thr
dead_mask = xcorr_neighbors_med < dead_channel_thr
out_mask  = xcorr_distant_med < outside_channel_thr

print(f"Noise by HF PSD only: {np.sum(noise_hf)} / {len(noise_hf)}")
print(f"Noise by neighbor-coh only: {np.sum(noise_coh)} / {len(noise_coh)}")
print(f"Dead by neighbor-coh: {np.sum(dead_mask)} / {len(dead_mask)}")
print(f'Out-of-brain by distant-coh: {np.sum(out_mask)} / {len(out_mask)}')

# -----------------------------
# Plots
# -----------------------------
idx = np.arange(len(psd_hf_med))

fig, axs = plt.subplots(2, 2, figsize=(11, 9))

# 1) HF PSD across channels
axs[0,0].plot(idx, psd_hf_med, lw=1)
axs[0,0].axhline(psd_hf_threshold, ls='--')
axs[0,0].set_title("Mean PSD in HF band (> nyquist_threshold * Nyquist)")
axs[0,0].set_xlabel("channel index")
axs[0,0].set_ylabel("μV²/Hz")

# 2) Neighbor “coherence” proxy
axs[0,1].plot(idx, xcorr_neighbors_med, lw=1)
axs[0,1].axhline(noisy_channel_thr, ls='--', label='noisy_thr')
axs[0,1].axhline(dead_channel_thr,  ls='--', label='dead_thr')
axs[0,1].set_title("Neighbor similarity (IBL xcorr_neighbors)")
axs[0,1].set_xlabel("channel index")
axs[0,1].legend(loc='best')

# 3) Distant “coherence” proxy (tail)
axs[1,0].plot(idx, xcorr_distant_med, lw=1)
axs[1,0].axhline(outside_channel_thr, ls='--')
axs[1,0].set_title("Distant similarity (IBL xcorr_distant)")
axs[1,0].set_xlabel("channel index")

# 4) Decision space: HF PSD vs neighbor similarity
axs[1,1].scatter(psd_hf_med, xcorr_neighbors_med, s=8, alpha=0.7)
axs[1,1].axvline(psd_hf_threshold, ls='--')
axs[1,1].axhline(noisy_channel_thr, ls='--')
axs[1,1].set_xlabel("HF mean PSD (μV²/Hz)")
axs[1,1].set_ylabel("xcorr_neighbors")
axs[1,1].set_title("Noise region: right of vertical OR above horizontal")

plt.tight_layout()
plt.show()

# Final useful arrays you can inspect:
#   labels_mode              : 0 good, 1 dead, 2 noise, 3 out  (SI's across-chunk decision)
#   psd_hf_med               : per-channel mean HF PSD (μV²/Hz)
#   xcorr_neighbors_med      : per-channel neighbor similarity
#   xcorr_distant_med        : per-channel distant similarity


In [ ]:
rec1.has_scaleable_traces()
rec1.get_channel_gains()[:10]

#### 1. raw_rec

In [ ]:
from scipy.signal import welch, coherence
from sklearn.neighbors import NearestNeighbors

# ---- inputs ----
# replace this with your actual RecordingExtractor:
rec_used = raw_rec # <-- ensure this exists

# ---- configuration ----
# 1) Downsample then bandpass
rec_ds = spre.resample(rec_used, resample_rate=10_000)  # 10 kHz
rec_bp = spre.bandpass_filter(rec_ds, 300, 4_000)

# 2) Snippet lengths
PSD_SECONDS = 20
COH_SECONDS = 10

# 3) Welch params
N_PERSEG = 1024
N_OVERLAP = 512

# 4) Channel bookkeeping
fs = rec_bp.get_sampling_frequency()
channel_ids = np.array(rec_bp.channel_ids)
nch = rec_bp.get_num_channels()

# 5) Optional subset for coherence
subset = np.arange(0, nch, 8)  # ~every 8th channel

# --- PSD per channel ---
def psd_per_channel(recording, nperseg=2048, noverlap=1024, max_seconds=120, segment_index=0):
    n_samples = min(int(max_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    f, Pxx = welch(X, fs=recording.get_sampling_frequency(),
                   nperseg=nperseg, noverlap=noverlap, axis=0)
    return f, Pxx  # (F,), (F, n_channels)

f_psd, Pxx = psd_per_channel(rec_bp, nperseg=N_PERSEG, noverlap=N_OVERLAP, max_seconds=PSD_SECONDS)

# --- simple PSD features ---
def band_power(f, Pxx, f_lo, f_hi):
    # guard upper bound by Nyquist in f
    f_hi = min(f_hi, float(f.max()))
    m = (f >= f_lo) & (f <= f_hi)
    if not np.any(m):
        return np.zeros(Pxx.shape[1], dtype=float)
    return np.trapz(Pxx[m, :], f[m], axis=0)

bp_low = band_power(f_psd, Pxx, 0.5, 1.0)         # “drift”
bp_spk = band_power(f_psd, Pxx, 300, 4_000)       # spike band (respect 10 kHz Fs)
mains = 60.0; bw = 1.0
mains_power = band_power(f_psd, Pxx, mains-bw, mains+bw)
rms = np.sqrt(np.trapz(Pxx, f_psd, axis=0))       # broadband RMS from PSD

# --- neighbors from probe geometry ---
geom = rec_bp.get_channel_locations()             # (nch, 2) or (nch, 3)
nbrs = NearestNeighbors(n_neighbors=min(5, nch), algorithm='ball_tree').fit(geom)
distances, indices = nbrs.kneighbors(geom)        # indices[i][0] is i itself

# --- coherence helpers ---
def avg_neighbor_coherence(recording, indices, win_seconds=10, nperseg=1024, noverlap=512, segment_index=0):
    n_samples = min(int(win_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    fs = recording.get_sampling_frequency()
    n_ch = X.shape[1]
    avg_coh = np.full(n_ch, np.nan, dtype=float)
    for i in range(n_ch):
        neighs = indices[i][1:]  # skip self
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            m = (f >= 300) & (f <= min(4_000, f.max()))
            cvals.append(np.nanmean(Cij[m]) if np.any(m) else np.nan)
        avg_coh[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return avg_coh

def avg_neighbor_coherence_subset(recording, indices, subset, **kw):
    """Compute avg coherence only for the channels in `subset` to save time."""
    out = np.full(recording.get_num_channels(), np.nan, dtype=float)
    n_samples = min(int(kw.get('win_seconds', 10) * recording.get_sampling_frequency()),
                    recording.get_num_samples(kw.get('segment_index', 0)))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=kw.get('segment_index', 0))
    fs = recording.get_sampling_frequency()
    nperseg = kw.get('nperseg', 1024)
    noverlap = kw.get('noverlap', 512)

    # precompute frequency mask once using a reference pair to get f-grid
    f_ref, _ = coherence(X[:, subset[0]], X[:, indices[subset[0]][1]], fs=fs, nperseg=nperseg, noverlap=noverlap)
    fmask = (f_ref >= 300) & (f_ref <= min(4_000, f_ref.max()))

    for i in subset:
        neighs = indices[i][1:]
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            cvals.append(np.nanmean(Cij[fmask]) if np.any(fmask) else np.nan)
        out[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return out

# Choose ONE of the two:
avg_coh = avg_neighbor_coherence(rec_bp, indices, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)
# avg_coh = avg_neighbor_coherence_subset(rec_bp, indices, subset, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)

# --- heuristic flags ---
flags = {
    "flat_low_rms":  rms < np.percentile(rms, 2),
    "no_spike_band": bp_spk < np.percentile(bp_spk, 2),
    "huge_line_noise": mains_power > np.percentile(mains_power, 98),
    "low_coherence":  avg_coh < np.nanpercentile(avg_coh, 5),
}
bad_any = np.zeros(nch, dtype=bool)
for v in flags.values():
    # careful: comparisons with NaN are False; we keep them unflagged unless other criteria hit
    bad_any |= np.nan_to_num(v, nan=False)

print(f"Heuristic bad channels: {np.where(bad_any)[0].size}/{nch}")

In [ ]:
# --- plots ---
fig, axs = plt.subplots(2, 2, figsize=(11, 9))

axs[0,0].hist(rms, bins=40)
axs[0,0].set_title("Broadband RMS (from PSD) per channel")

axs[0,1].scatter(bp_spk, mains_power, s=6, alpha=0.7)
axs[0,1].set_xlabel("Spike-band power (300–4000 Hz)")
axs[0,1].set_ylabel(f"{int(mains)} Hz power")
axs[0,1].set_title("Line-noise vs spike-band")
axs[0,1].axhline(np.percentile(mains_power, 98), ls="--")

order = np.argsort(np.nan_to_num(avg_coh, nan=-1))
axs[1,0].plot(avg_coh[order], lw=1)
axs[1,1].set_xlim(0, 384)
axs[1,0].set_title("Avg neighbor coherence (300–4000 Hz)")
axs[1,0].set_xlabel("channels (sorted)")
axs[1,0].set_ylabel("coherence")

good_idx = np.where(~bad_any)[0][:5]
bad_idx  = np.where(bad_any)[0][:5]
for ch in good_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.5)
for ch in bad_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.7, linestyle='--')
axs[1,1].set_xlim(1, min(8_000, f_psd.max()))
axs[1,1].set_xscale('log')
axs[1,1].set_title("PSDs: solid=good, dashed=flagged")
axs[1,1].set_xlabel("Hz")
axs[1,1].set_ylabel("dB/Hz")

plt.tight_layout()
plt.show()


#### 2. rec1 after 400 Hz filtering

In [ ]:
from scipy.signal import welch, coherence
from sklearn.neighbors import NearestNeighbors

# ---- inputs ----
# replace this with your actual RecordingExtractor:
rec_used = rec1 # <-- ensure this exists

# ---- configuration ----
# 1) Downsample then bandpass
rec_ds = spre.resample(rec_used, resample_rate=10_000)  # 10 kHz
rec_bp = spre.bandpass_filter(rec_ds, 300, 4_000)

# 2) Snippet lengths
PSD_SECONDS = 20
COH_SECONDS = 10

# 3) Welch params
N_PERSEG = 1024
N_OVERLAP = 512

# 4) Channel bookkeeping
fs = rec_bp.get_sampling_frequency()
channel_ids = np.array(rec_bp.channel_ids)
nch = rec_bp.get_num_channels()

# 5) Optional subset for coherence
subset = np.arange(0, nch, 8)  # ~every 8th channel

# --- PSD per channel ---
def psd_per_channel(recording, nperseg=2048, noverlap=1024, max_seconds=120, segment_index=0):
    n_samples = min(int(max_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    f, Pxx = welch(X, fs=recording.get_sampling_frequency(),
                   nperseg=nperseg, noverlap=noverlap, axis=0)
    return f, Pxx  # (F,), (F, n_channels)

f_psd, Pxx = psd_per_channel(rec_bp, nperseg=N_PERSEG, noverlap=N_OVERLAP, max_seconds=PSD_SECONDS)

# --- simple PSD features ---
def band_power(f, Pxx, f_lo, f_hi):
    # guard upper bound by Nyquist in f
    f_hi = min(f_hi, float(f.max()))
    m = (f >= f_lo) & (f <= f_hi)
    if not np.any(m):
        return np.zeros(Pxx.shape[1], dtype=float)
    return np.trapz(Pxx[m, :], f[m], axis=0)

bp_low = band_power(f_psd, Pxx, 0.5, 1.0)         # “drift”
bp_spk = band_power(f_psd, Pxx, 300, 4_000)       # spike band (respect 10 kHz Fs)
mains = 60.0; bw = 1.0
mains_power = band_power(f_psd, Pxx, mains-bw, mains+bw)
rms = np.sqrt(np.trapz(Pxx, f_psd, axis=0))       # broadband RMS from PSD

# --- neighbors from probe geometry ---
geom = rec_bp.get_channel_locations()             # (nch, 2) or (nch, 3)
nbrs = NearestNeighbors(n_neighbors=min(5, nch), algorithm='ball_tree').fit(geom)
distances, indices = nbrs.kneighbors(geom)        # indices[i][0] is i itself

# --- coherence helpers ---
def avg_neighbor_coherence(recording, indices, win_seconds=10, nperseg=1024, noverlap=512, segment_index=0):
    n_samples = min(int(win_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    fs = recording.get_sampling_frequency()
    n_ch = X.shape[1]
    avg_coh = np.full(n_ch, np.nan, dtype=float)
    for i in range(n_ch):
        neighs = indices[i][1:]  # skip self
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            m = (f >= 300) & (f <= min(4_000, f.max()))
            cvals.append(np.nanmean(Cij[m]) if np.any(m) else np.nan)
        avg_coh[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return avg_coh

def avg_neighbor_coherence_subset(recording, indices, subset, **kw):
    """Compute avg coherence only for the channels in `subset` to save time."""
    out = np.full(recording.get_num_channels(), np.nan, dtype=float)
    n_samples = min(int(kw.get('win_seconds', 10) * recording.get_sampling_frequency()),
                    recording.get_num_samples(kw.get('segment_index', 0)))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=kw.get('segment_index', 0))
    fs = recording.get_sampling_frequency()
    nperseg = kw.get('nperseg', 1024)
    noverlap = kw.get('noverlap', 512)

    # precompute frequency mask once using a reference pair to get f-grid
    f_ref, _ = coherence(X[:, subset[0]], X[:, indices[subset[0]][1]], fs=fs, nperseg=nperseg, noverlap=noverlap)
    fmask = (f_ref >= 300) & (f_ref <= min(4_000, f_ref.max()))

    for i in subset:
        neighs = indices[i][1:]
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            cvals.append(np.nanmean(Cij[fmask]) if np.any(fmask) else np.nan)
        out[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return out

# Choose ONE of the two:
# avg_coh = avg_neighbor_coherence(rec_bp, indices, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)
avg_coh = avg_neighbor_coherence_subset(rec_bp, indices, subset, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)

# --- heuristic flags ---
flags = {
    "flat_low_rms":  rms < np.percentile(rms, 2),
    "no_spike_band": bp_spk < np.percentile(bp_spk, 2),
    "huge_line_noise": mains_power > np.percentile(mains_power, 98),
    "low_coherence":  avg_coh < np.nanpercentile(avg_coh, 5),
}
bad_any = np.zeros(nch, dtype=bool)
for v in flags.values():
    # careful: comparisons with NaN are False; we keep them unflagged unless other criteria hit
    bad_any |= np.nan_to_num(v, nan=False)

print(f"Heuristic bad channels: {np.where(bad_any)[0].size}/{nch}")

In [ ]:
# --- plots ---
fig, axs = plt.subplots(2, 2, figsize=(11, 9))

axs[0,0].hist(rms, bins=40)
axs[0,0].set_title("Broadband RMS (from PSD) per channel")

axs[0,1].scatter(bp_spk, mains_power, s=6, alpha=0.7)
axs[0,1].set_xlabel("Spike-band power (300–4000 Hz)")
axs[0,1].set_ylabel(f"{int(mains)} Hz power")
axs[0,1].set_title("Line-noise vs spike-band")
axs[0,1].axhline(np.percentile(mains_power, 98), ls="--")

order = np.argsort(np.nan_to_num(avg_coh, nan=-1))
axs[1,0].plot(avg_coh[order], lw=1)
axs[1,1].set_xlim(0, 384)
axs[1,0].set_title("Avg neighbor coherence (300–4000 Hz)")
axs[1,0].set_xlabel("channels (sorted)")
axs[1,0].set_ylabel("coherence")

good_idx = np.where(~bad_any)[0][:5]
bad_idx  = np.where(bad_any)[0][:5]
for ch in good_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.5)
for ch in bad_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.7, linestyle='--')
axs[1,1].set_xlim(1, min(8_000, f_psd.max()))
axs[1,1].set_xscale('log')
axs[1,1].set_title("PSDs: solid=good, dashed=flagged")
axs[1,1].set_xlabel("Hz")
axs[1,1].set_ylabel("dB/Hz")

plt.tight_layout()
plt.show()


#### 2. after CMR

In [ ]:
from scipy.signal import welch, coherence
from sklearn.neighbors import NearestNeighbors

# ---- inputs ----
# replace this with your actual RecordingExtractor:
rec_used = rec4  # <-- ensure this exists

# ---- configuration ----
# 1) Downsample then bandpass
rec_ds = spre.resample(rec_used, resample_rate=10_000)  # 10 kHz
rec_bp = spre.bandpass_filter(rec_ds, 300, 4_000)

# 2) Snippet lengths
PSD_SECONDS = 20
COH_SECONDS = 10

# 3) Welch params
N_PERSEG = 1024
N_OVERLAP = 512

# 4) Channel bookkeeping
fs = rec_bp.get_sampling_frequency()
channel_ids = np.array(rec_bp.channel_ids)
nch = rec_bp.get_num_channels()

# 5) Optional subset for coherence
subset = np.arange(0, nch, 8)  # ~every 8th channel

# --- PSD per channel ---
def psd_per_channel(recording, nperseg=2048, noverlap=1024, max_seconds=120, segment_index=0):
    n_samples = min(int(max_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    f, Pxx = welch(X, fs=recording.get_sampling_frequency(),
                   nperseg=nperseg, noverlap=noverlap, axis=0)
    return f, Pxx  # (F,), (F, n_channels)

f_psd, Pxx = psd_per_channel(rec_bp, nperseg=N_PERSEG, noverlap=N_OVERLAP, max_seconds=PSD_SECONDS)

# --- simple PSD features ---
def band_power(f, Pxx, f_lo, f_hi):
    # guard upper bound by Nyquist in f
    f_hi = min(f_hi, float(f.max()))
    m = (f >= f_lo) & (f <= f_hi)
    if not np.any(m):
        return np.zeros(Pxx.shape[1], dtype=float)
    return np.trapz(Pxx[m, :], f[m], axis=0)

bp_low = band_power(f_psd, Pxx, 0.5, 1.0)         # “drift”
bp_spk = band_power(f_psd, Pxx, 300, 4_000)       # spike band (respect 10 kHz Fs)
mains = 60.0; bw = 1.0
mains_power = band_power(f_psd, Pxx, mains-bw, mains+bw)
rms = np.sqrt(np.trapz(Pxx, f_psd, axis=0))       # broadband RMS from PSD

# --- neighbors from probe geometry ---
geom = rec_bp.get_channel_locations()             # (nch, 2) or (nch, 3)
nbrs = NearestNeighbors(n_neighbors=min(5, nch), algorithm='ball_tree').fit(geom)
distances, indices = nbrs.kneighbors(geom)        # indices[i][0] is i itself

# --- coherence helpers ---
def avg_neighbor_coherence(recording, indices, win_seconds=10, nperseg=1024, noverlap=512, segment_index=0):
    n_samples = min(int(win_seconds * recording.get_sampling_frequency()),
                    recording.get_num_samples(segment_index))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=segment_index)
    fs = recording.get_sampling_frequency()
    n_ch = X.shape[1]
    avg_coh = np.full(n_ch, np.nan, dtype=float)
    for i in range(n_ch):
        neighs = indices[i][1:]  # skip self
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            m = (f >= 300) & (f <= min(4_000, f.max()))
            cvals.append(np.nanmean(Cij[m]) if np.any(m) else np.nan)
        avg_coh[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return avg_coh

def avg_neighbor_coherence_subset(recording, indices, subset, **kw):
    """Compute avg coherence only for the channels in `subset` to save time."""
    out = np.full(recording.get_num_channels(), np.nan, dtype=float)
    n_samples = min(int(kw.get('win_seconds', 10) * recording.get_sampling_frequency()),
                    recording.get_num_samples(kw.get('segment_index', 0)))
    X = recording.get_traces(start_frame=0, end_frame=n_samples, segment_index=kw.get('segment_index', 0))
    fs = recording.get_sampling_frequency()
    nperseg = kw.get('nperseg', 1024)
    noverlap = kw.get('noverlap', 512)

    # precompute frequency mask once using a reference pair to get f-grid
    f_ref, _ = coherence(X[:, subset[0]], X[:, indices[subset[0]][1]], fs=fs, nperseg=nperseg, noverlap=noverlap)
    fmask = (f_ref >= 300) & (f_ref <= min(4_000, f_ref.max()))

    for i in subset:
        neighs = indices[i][1:]
        cvals = []
        for j in neighs:
            f, Cij = coherence(X[:, i], X[:, j], fs=fs, nperseg=nperseg, noverlap=noverlap)
            cvals.append(np.nanmean(Cij[fmask]) if np.any(fmask) else np.nan)
        out[i] = np.nanmean(cvals) if len(cvals) else np.nan
    return out

# Choose ONE of the two:
# avg_coh = avg_neighbor_coherence(rec_bp, indices, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)
avg_coh = avg_neighbor_coherence_subset(rec_bp, indices, subset, win_seconds=COH_SECONDS, nperseg=N_PERSEG, noverlap=N_OVERLAP)

# --- heuristic flags ---
flags = {
    "flat_low_rms":  rms < np.percentile(rms, 2),
    "no_spike_band": bp_spk < np.percentile(bp_spk, 2),
    "huge_line_noise": mains_power > np.percentile(mains_power, 98),
    "low_coherence":  avg_coh < np.nanpercentile(avg_coh, 5),
}
bad_any = np.zeros(nch, dtype=bool)
for v in flags.values():
    # careful: comparisons with NaN are False; we keep them unflagged unless other criteria hit
    bad_any |= np.nan_to_num(v, nan=False)

print(f"Heuristic bad channels: {np.where(bad_any)[0].size}/{nch}")

# --- plots ---
fig, axs = plt.subplots(2, 2, figsize=(11, 9))

axs[0,0].hist(rms, bins=40)
axs[0,0].set_title("Broadband RMS (from PSD) per channel")

axs[0,1].scatter(bp_spk, mains_power, s=6, alpha=0.7)
axs[0,1].set_xlabel("Spike-band power (300–4000 Hz)")
axs[0,1].set_ylabel(f"{int(mains)} Hz power")
axs[0,1].set_title("Line-noise vs spike-band")
axs[0,1].axhline(np.percentile(mains_power, 98), ls="--")

order = np.argsort(np.nan_to_num(avg_coh, nan=-1))
axs[1,0].plot(avg_coh[order], lw=1)
axs[1,0].set_title("Avg neighbor coherence (300–4000 Hz)")
axs[1,0].set_xlabel("channels (sorted)")
axs[1,0].set_ylabel("coherence")

good_idx = np.where(~bad_any)[0][:5]
bad_idx  = np.where(bad_any)[0][:5]
for ch in good_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.5)
for ch in bad_idx:
    axs[1,1].plot(f_psd, 10*np.log10(Pxx[:, ch]), alpha=0.7, linestyle='--')
axs[1,1].set_xlim(1, min(8_000, f_psd.max()))
axs[1,1].set_xscale('log')
axs[1,1].set_title("PSDs: solid=good, dashed=flagged")
axs[1,1].set_xlabel("Hz")
axs[1,1].set_ylabel("dB/Hz")

plt.tight_layout()
plt.show()


In [ ]:
if skipOptional == 0:
    %matplotlib widget
    si.plot_traces({'filter':rec2, 'cmr': rec4}, backend='ipywidgets')

In [ ]:
pd.set_option('display.max_rows',30)
channel_by_shank=pd.DataFrame()
channel_by_shank['shank_ids'] = df['shank_ids']
channel_by_shank['channel_ids']=df.index
channel_by_shank['channel_ids'] = channel_by_shank['channel_ids'].apply(lambda x: f'imec0.ap#AP{x}')
# channel_by_shank

In [ ]:
unique_shank_ids = channel_by_shank['shank_ids'].unique()

fig, axs = plt.subplots(ncols=len(unique_shank_ids), figsize=(20, 10))

for i, shank_id in enumerate(unique_shank_ids):
    shank_channels = channel_by_shank[channel_by_shank['shank_ids'] == shank_id]['channel_ids'].tolist()
    si.plot_traces(rec1, channel_ids=shank_channels, backend='matplotlib', clim=(-50, 50), ax=axs[i], show_channel_ids=True)

In [ ]:
# here we use static plot using matplotlib backend
fig, axs = plt.subplots(ncols=3, figsize=(20, 10))

si.plot_traces(rec1, backend='matplotlib',  clim=(-50, 50), ax=axs[0])
si.plot_traces(rec4, backend='matplotlib',  clim=(-50, 50), ax=axs[1])
si.plot_traces(rec, backend='matplotlib',  clim=(-50, 50), ax=axs[2])
for i, label in enumerate(('filter', 'cmr', 'final')):
    axs[i].set_title(label)

In [ ]:
if skipOptional == 0:
    # plot some channels
    fig, ax = plt.subplots(figsize=(20, 10))
    some_chans = rec.channel_ids[[100, 150, 200, ]]
    si.plot_traces({'filter':rec1, 'cmr': rec4}, backend='matplotlib', mode='line', ax=ax, channel_ids=some_chans)

## We will not be saving the preprocessed file but we could...

## Next, let's check for noise levels

In [ ]:
rec = rec4

## Detect and localize peaks in your recording

### We wil look for peaks and check for drifts prior to sorting.

In [ ]:
if skipOptional == 0:
    from spikeinterface.sortingcomponents.peak_detection import detect_peaks # usually takes 35min

    job_kwargs = dict(n_jobs=40, chunk_duration='1s', progress_bar=True)
    peaks = detect_peaks(rec,  method='locally_exclusive', noise_levels=noise_levels_int16,
                        detect_threshold=5, radius_um=50., **job_kwargs)
    peaks

In [ ]:
if skipOptional == 0:

    from spikeinterface.sortingcomponents.peak_localization import localize_peaks # usually takes 19min

    peak_locations = localize_peaks(rec, peaks, method='center_of_mass', radius_um=50., **job_kwargs)

## Let's check for drift

In [ ]:
if skipOptional == 0:
    # check for drifts
    fs = rec.sampling_frequency
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(peaks['sample_index'] / fs, peak_locations['y'], color='k', marker='.',  alpha=0.002)

### You can also check where on the probe your peak estimates are located:

In [ ]:
if skipOptional == 0:
    fig, ax = plt.subplots(figsize=(15, 10))
    si.plot_probe_map(rec, ax=ax, with_channel_ids=True)
    ax.set_ylim(-100, 150)

    ax.scatter(peak_locations['x'], peak_locations['y'], color='purple', alpha=0.002)

# Now that we've checked our recording quality, we can run a spike sorter. Here, we will be running KS-2.5

#### First, we need to tell spikeinterface where to find Kilosort

In [ ]:
from spikeinterface.sorters import Kilosort2_5Sorter

Kilosort2_5Sorter.set_kilosort2_5_path(r'C:\Users\janet\Documents\GitHub\Kilosort-2.5.2')

In [ ]:
#This reflects local installs and not docker containers
si.installed_sorters()

### Check the default params

In [ ]:
si.get_default_sorter_params('kilosort2_5')

### Run the sorter

In [ ]:
# run kilosort2.5 with drift correction # usually takes 70min on predator and 35min on alienware
kilosort_output_folder = output_folder / 'kilosort2.5_output'

# Define the parameters for running kilosort2.5
params_kilosort2_5 = {'do_correction': True}

# Check if the kilosort output folder exists
if kilosort_output_folder.exists() and kilosort_output_folder.is_dir():
    # Read the sorting results from the existing folder
    sorting = si.read_sorter_folder(kilosort_output_folder)
else:
    # Run the kilosort2.5 sorter with drift correction
    sorting = si.run_sorter('kilosort2_5', rec, output_folder=kilosort_output_folder, remove_existing_folder=True,
                            verbose=True, **params_kilosort2_5)

# Print the number of detected units
print('Your session has {} detected units'.format(len(sorting.get_unit_ids())))

In [ ]:
# # run kilosort2.5 with drift correction # usually takes 70min on predator and 35min on alienware
# params_kilosort2_5 = {'do_correction': True}

# sorting = si.run_sorter('kilosort2_5', rec, output_folder=output_folder / 'kilosort2.5_output', remove_existing_folder=True,
#                     verbose=True, **params_kilosort2_5)

In [ ]:
# # the results can be read back for futur session
# sorting = si.read_sorter_folder(output_folder / 'kilosort2.5_output')
# print('Your session has {} detected units'.format(sorting))

In [ ]:
# spike_vector = sorting.to_spike_vector()
# len(spike_vector)

# Post processing your sorted units

### All postprocessing is dependent on the created of the WaveformExtractor object. This object combines a `recording` and a `sorting` object and extracts some waveform snippets (500 by default) for each units.

### *Note that we use the `sparse=True` option. This option is important because the waveforms will be extracted only for a few channels around the main channel of each unit. This saves tons ### of disk space and speeds up the waveforms extraction and further processing.

In [ ]:
job_kwargs = dict(n_jobs=-1, chunk_duration='1s', progress_bar=True)

In [ ]:
print(rec)
print(sorting)

In [ ]:
sorting_removed_excess_spikes = si.remove_excess_spikes(sorting, rec) # needed for some sessions

In [ ]:
waveforms_folder = output_folder / 'waveforms_kilosort2.5'

# Define the parameters for waveform extraction
waveform_params = {
    'sparse': True,
    'max_spikes_per_unit': 500,
    'ms_before': 1.5,
    'ms_after': 2.0,
}

# Check if the waveforms folder exists
if waveforms_folder.exists() and waveforms_folder.is_dir():
    # Load the waveform extractor from the existing folder
    we = si.load_waveforms(waveforms_folder)
else:
    # Extract the waveforms and save them to the specified folder
    we = si.extract_waveforms(rec, sorting_removed_excess_spikes, folder=waveforms_folder, **waveform_params, **job_kwargs)

# Print the waveform extractor and its folder
print(we)
print(we.folder)

## Remove duplicate units and save the sorted object without redundant units. Here, the output is a sorting object.

In [ ]:
if dup_removed == 0:
    sorting = sorting_removed_excess_spikes
else: 
    sorting = si.remove_redundant_units(we, align=True, delta_time=0.4, agreement_threshold=0.2,
                            duplicate_threshold=0.8, remove_strategy='minimum_shift')

In [ ]:
if skipOptional == 0:
    # Retrieve waveforms 
    unit_ids = sorting.unit_ids

    for unit_id in unit_ids:
        wfs = we.get_waveforms(unit_id)
        print(unit_id, ':', wfs.shape)

In [ ]:
if skipOptional == 0:
    # View WaveForm templates, here we have the first 3
    number_of_units = 3

    for unit_id in unit_ids[:number_of_units]:
        fig, ax = plt.subplots()
        template = we.get_template(unit_id=unit_id, mode='median')
        print(template.shape)
        ax.plot(template)
        ax.set_title(f'{unit_id}')

### There are also extensions the `WaveExtractor` module please refer to the documentation if you wish to use these!

### Now, we can summarize various analyses such as noise levels, correlograms, unit locations, etc.

In [ ]:
# this is the older version
if skipOptional == 0:
    noise_levels = si.compute_noise_levels(we)
    corrGram = si.compute_correlograms(we)
    unitLoc = si.compute_unit_locations(we)
    unitAmp = si.compute_spike_amplitudes(we, **job_kwargs)
    simTemp = si.compute_template_similarity(we)
    # isiHist = si.compute_isi_histograms(sorting, load_if_exists=False, window_ms = 50,
    #                                     bin_ms=1, method='auto' ) # took 26 min on predator and 17min on alienware

In [ ]:
# analyzer_folder = (output_folder /"analyzer")
# analyzer = create_sorting_analyzer(sorting=sorting, recording=rec, format="binary_folder",folder=analyzer_folder)
# print(analyzer) 
# # took 54 min on alienware

In [ ]:
analyzer_folder = (output_folder /"analyzer")
if analyzer_folder.exists() and analyzer_folder.is_dir():
    analyzer = load_sorting_analyzer(analyzer_folder)
else:
    analyzer = create_sorting_analyzer(sorting=sorting, recording=rec, format="binary_folder", folder=analyzer_folder)
print(analyzer)

In [ ]:
analyzer.compute('correlograms')

In [ ]:
isiHist = si.compute_isi_histograms(analyzer, load_if_exists=False, window_ms = 50,
                                    bin_ms=1, method='auto' ) 

In [ ]:
if skipOptional == 0:
    # You can even run PCA analysis on the extracted waveforms
    we = si.extract_waveforms(rec, sorting, folder='waveforms')
    pc = si.compute_principal_components(we, n_components=3, mode='by_channel_local')
    # get pre-computed projections for unit_id=1
    projections = pc.get_projections(unit_id=1)
    # get all pre-computed projections and labels
    all_projections, all_labels = pc.get_all_projections()
    # retrieve fitted pca model(s)
    pca_model = pc.get_pca_model()
    # compute projections on new waveforms
    proj_new = pc.project_new(new_waveforms)
    # run for all spikes in the SortingExtractor
    pc.run_for_all_spikes(file_path="all_pca_projections.npy")

# QC metrics

### We have a single function `compute_quality_metrics(WaveformExtractor)` that returns a pandas.Dataframe with the desired metrics.

### Please visit the metrics documentation for more information and a list of all supported metrics.

### Some metrics are based on PCA (like 'isolation_distance', 'l_ratio', 'd_prime') and require to estimate PCA for their computation. This can be achieved with:

### `si.compute_principal_components(waveform_extractor)`

In [ ]:
#Lists all available QC metrics
#si.get_quality_metric_list()

In [ ]:
metrics = si.compute_quality_metrics(we, metric_names=['firing_rate', 'presence_ratio', 'snr',
                                                       'isi_violation', 'amplitude_cutoff'])
metrics

# Curation using metrics
### A very common curation approach is to threshold these metrics to select good units:

In [ ]:
amplitude_cutoff_thresh = 0.1
isi_violations_ratio_thresh = 1
presence_ratio_thresh = 0.9

our_query = f"(amplitude_cutoff < {amplitude_cutoff_thresh}) & (isi_violations_ratio < {isi_violations_ratio_thresh}) & (presence_ratio > {presence_ratio_thresh})"
print(our_query)

In [ ]:
keep_units = metrics.query(our_query)
keep_unit_ids = keep_units.index.values
keep_unit_ids.shape

# Export and Save our results 
### In order to export the final results we need to make a copy of the the waveforms, but only for the selected units (so we can avoid to compute them again).

In [ ]:
# we_clean = we.select_units(keep_unit_ids, new_folder=output_folder / 'waveforms_clean') # no longer works in the new version of spikeInterface
we_clean = we.select_units(keep_unit_ids)
we_clean

In [ ]:
# export spike sorting report to a folder
si.export_report(we_clean, output_folder / 'report', format='png') # took 30 min on predator and 13min on alienware

In [ ]:
if skipOptional == 0:
    we_clean = si.load_waveforms(output_folder / 'waveforms_clean')
    we_clean

In [ ]:
#si.plot_sorting_summary(we_clean, backend='sortingview') # does not work

In [ ]:
# potentially helpful: re-load data after phy sorting
# sorting_phy = se.PhySortingExtractor('path-to-phy-folder', exclude_cluster_groups=['noise'])